# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll leverage Croissant's schema to explore available record sets and fields via their unique `@id`s, and perform preliminary data analysis.

### Dataset Source
The dataset is defined via a Croissant schema and is accessible here:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed (restart the kernel after installation if needed)
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This will help us overview the available resources and plan our exploration accordingly.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata and Croissant structure
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s as defined in the Croissant schema.
This ensures you use precise entity references in all further work.

In [ ]:
# List all record sets and their fields using their @id
print("Available record sets (by `@id`):\n")
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("- No record sets declared in metadata. Attempting to discover from distribution files...\n")
    
    # List available distributions (data files)
    if hasattr(dataset.metadata, 'distribution'):
        for dist in dataset.metadata.distribution:
            print(f"Distribution `@id`: {dist.get('@id', dist)}")
else:
    for rs in record_sets:
        print(f"- Record set `@id`: {rs['@id']}")
        if 'field' in rs:
            for field in rs['field']:
                print(f"    - Field `@id`: {field['@id']} | name: {field.get('name', '')}")

## 3. Data Extraction
Extract data from a specific record set or data table.

👉 **Note:** The Croissant schema for this FAIR² dataset does not explicitly define top-level record sets with `@id`, but it does provide two main distributions (data files). Their `@id`s are used for data extraction below.

In [ ]:
# List the available distributions (data files) and treat each as a record set.
distributions = dataset.metadata.distribution
record_set_ids = []

print("Available data resource `@id`s for extraction:")
for dist in distributions:
    rid = dist['@id'] if isinstance(dist, dict) and '@id' in dist else dist
    print(f"- {rid}")
    record_set_ids.append(rid)

# We'll extract all available tables
dataframes = {}
for rid in record_set_ids:
    print(f"Loading records from: {rid}")
    try:
        records = list(dataset.records(record_set=rid))
        df = pd.DataFrame(records)
        print(f"Columns in this record set: {df.columns.tolist()}")
        print(df.head(2))
        dataframes[rid] = df
    except Exception as e:
        print(f"  [Warning] Could not load records from {rid}: {e}")

## 4. Exploratory Data Analysis (EDA)

We'll perform EDA on one of the main dataframes.
- We'll programmatically select one of the dataframes, pick a numeric column by its `@id`, filter outliers, normalize values, and group by a categorical field (using field `@id`).

> ⚠️ Replace `<numeric_field_id>` and `<group_field_id>` below with the actual column names or Croissant field IDs revealed above.

In [ ]:
# Pick the first record set (data table)
target_record_set_id = record_set_ids[0]  # or choose the relevant one as printed above
df = dataframes[target_record_set_id]

print(f"Columns in the selected record set: {df.columns.tolist()}")

# Let's try to infer a numeric field and a group field (@id or column names)
numeric_candidates = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
group_candidates = df.select_dtypes(include=['object']).columns.tolist()

# Display candidates
print(f"\nNumeric field candidates: {numeric_candidates}")
print(f"Group field candidates: {group_candidates}")

# If columns are non-informative (e.g. unnamed), try other record set as needed
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # Use first numeric column discovered
else:
    raise ValueError("No obvious numeric field for filtering/normalization.")

group_field_id = group_candidates[0] if group_candidates else None

# EDA: filter, normalize, group
threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in "fi" else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records where {numeric_field_id} > {threshold:.1f} (threshold=mean):")
print(filtered_df.head())

filtered_df = filtered_df.copy()  # To avoid SettingWithCopy warning
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped by {group_field_id} (mean values):")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between numeric and categorical fields. Use column names or the field `@id`s from above.

The plot below illustrates the normalized values and group-wise averages.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field (original and normalized)
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(df[numeric_field_id], kde=True, ax=axes[0])
axes[0].set_title(f"Histogram of {numeric_field_id}")

sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, ax=axes[1], color='darkorange')
axes[1].set_title(f"Histogram of normalized {numeric_field_id} (filtered)")
plt.tight_layout()
plt.show()

# If grouping field is available, plot group means
if group_field_id and group_field_id in grouped_df.index:
    grouped_df[numeric_field_id].plot(kind='bar', title=f'Mean of {numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.show()

## 6. Conclusion

- This notebook demonstrated step-by-step loading, overview, and EDA for the FAIR² dataset using the `mlcroissant` library.
- All entities (dataframes, fields, record sets) are referenced by their Croissant `@id`, ensuring consistent access and future reproducibility.
- Further analysis can build upon these extracted and cleaned dataframes, leveraging metadata-driven selection of variables and deeper insights into knowledge adoption predictors in rangeland management practices in Northern Kenya.